对每个.npy文件进行降采样 (256x256x64 → 128x128x32)

In [ ]:
import os
import numpy as np
import nibabel as nib
from tqdm import tqdm
from scipy.ndimage import zoom
from sklearn.preprocessing import MinMaxScaler

def downsample_volume(volume, target_shape=(128, 128, 32)):
    """
    使用scipy.ndimage.zoom进行降采样
    """
    # 计算各维度的缩放因子
    factors = (
        target_shape[0] / volume.shape[0],
        target_shape[1] / volume.shape[1],
        target_shape[2] / volume.shape[2]
    )
    
    # 使用order=3进行三次样条插值
    downsampled = zoom(volume, factors, order=3)
    return downsampled

def process_directory(input_dir, output_dir, modality, target_shape=(128, 128, 32)):
    """
    处理单个模态目录中的所有npy文件
    """
    os.makedirs(output_dir, exist_ok=True)
    scaler = MinMaxScaler()
    
    files = sorted([f for f in os.listdir(input_dir) if f.endswith('.npy')])
    
    for filename in tqdm(files, desc=f'Processing {modality}'):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        # 加载数据
        volume = np.load(input_path)
        
        # 归一化 (MR需要，CT可能不需要)
        if modality == "MR":
            volume = scaler.fit_transform(volume.reshape(-1, volume.shape[-1])).reshape(volume.shape)
        
        # 降采样
        downsampled = downsample_volume(volume, target_shape)
        
        # 保存
        np.save(output_path, downsampled)

def main():
    original_root = "/home/featurize/data/Task1_Liu_split"
    new_root = "/home/featurize/data/Task1_Liu_split_128"
    
    # 处理训练集
    # for split in ['train', 'val', 'test']:  # 假设有这三个分割
    for split in ['train', 'val']:
        for modality in ['CT', 'MR']:
            input_dir = os.path.join(original_root, split, modality)
            output_dir = os.path.join(new_root, split, modality)
            
            print(f"Processing {input_dir} -> {output_dir}")
            process_directory(input_dir, output_dir, modality)

if __name__ == "__main__":
    main()

Processing /home/featurize/data/Task1_Liu_split/train/CT -> /home/featurize/data/Task1_Liu_split_128/train/CT


Processing CT: 100%|██████████| 144/144 [00:47<00:00,  3.06it/s]


Processing /home/featurize/data/Task1_Liu_split/train/MR -> /home/featurize/data/Task1_Liu_split_128/train/MR


Processing MR: 100%|██████████| 144/144 [00:37<00:00,  3.88it/s]


Processing /home/featurize/data/Task1_Liu_split/val/CT -> /home/featurize/data/Task1_Liu_split_128/val/CT


Processing CT: 100%|██████████| 36/36 [00:11<00:00,  3.18it/s]


Processing /home/featurize/data/Task1_Liu_split/val/MR -> /home/featurize/data/Task1_Liu_split_128/val/MR


Processing MR: 100%|██████████| 36/36 [00:10<00:00,  3.42it/s]

Processing /home/featurize/data/Task1_Liu_split/test/CT -> /home/featurize/data/Task1_Liu_split_128/test/CT


FileNotFoundError: [Errno 2] No such file or directory: '/home/featurize/data/Task1_Liu_split/test/CT'